# 02 — Run MiDaS on KITTI-C

Batch inference using `dpt_hybrid_384` over the KITTI-C manifest.
Predictions are saved as `.npy` files in `outputs/predictions/kitti_c/`.

**Requires:** manifest from notebook 01 and KITTI-C data in place.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'configs' / 'dataset_paths.yaml').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root from the current working directory')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
# --- Config ---
MODEL_TYPE = 'dpt_hybrid_384'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'kitti_c_manifest.csv'
PRED_DIR = PROJECT_ROOT / 'outputs' / 'predictions' / 'kitti_c'

# Set to a small number to test; set to None for full run
MAX_SAMPLES = None

In [3]:
import pandas as pd
from src.adapters.midas_adapter import MiDaSAdapter

df = pd.read_csv(MANIFEST_PATH, dtype={'frame_id': str})
df = df[df['image_path'].notna()].reset_index(drop=True)

if MAX_SAMPLES:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)

print(f'Running inference on {len(df)} images')

adapter = MiDaSAdapter(model_type=MODEL_TYPE)


/opt/conda/lib/python3.13/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Running inference on 63427 images


/opt/conda/lib/python3.13/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


Model loaded, number of parameters = 123M
[MiDaSAdapter] Model 'dpt_hybrid_384' loaded on cuda


In [4]:
import numpy as np
from tqdm.notebook import tqdm

BATCH_SIZE = 256   # increase if GPU memory allows (A100 80GB can go higher)
NUM_WORKERS = 4   # parallel CPU image-loading threads


def prediction_path(row):
    return (
        PRED_DIR
        / row['corruption_type']
        / str(row['severity'])
        / row['sequence']
        / f"{row['frame_id']}.npy"
    )

# Filter to images not yet predicted
todo_indices = [
    i for i, row in df.iterrows()
    if not prediction_path(row).exists()
]
todo_rows = df.loc[todo_indices].reset_index(drop=True)
skipped = len(df) - len(todo_rows)
print(f'Skipping {skipped} already-existing predictions. Running on {len(todo_rows)} images.')

image_paths = todo_rows['image_path'].tolist()

for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc='batches'):
    batch_paths = image_paths[start: start + BATCH_SIZE]
    batch_rows = todo_rows.iloc[start: start + BATCH_SIZE]

    depths = adapter.predict_batch(batch_paths, batch_size=len(batch_paths), num_workers=NUM_WORKERS)

    for depth, (_, row) in zip(depths, batch_rows.iterrows()):
        out_path = prediction_path(row)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(str(out_path), depth)

print('Done.')


Skipping 12952 already-existing predictions. Running on 50475 images.


batches:   0%|          | 0/198 [00:00<?, ?it/s]

Done.


## Next step

Run `03_eval_metrics.ipynb` to compute alignment and depth metrics.